# NB16 — Robust Regression

> **OLS squares residuals, so outliers dominate. Robust regression limits their influence.**

Three main approaches:
1. **Huber regression** — quadratic for small residuals, linear for large ones
2. **RANSAC** — random consensus: finds the line supported by the most inliers
3. **Theil-Sen** — median of all pairwise slopes: immune to 29.3% contamination


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, HuberRegressor, RANSACRegressor, TheilSenRegressor

np.random.seed(42)
n = 60
X = np.linspace(0, 10, n).reshape(-1,1)
y = 2*X.ravel() + 3 + np.random.normal(0, 1, n)

# Add 8 gross outliers
X_out = np.array([[1],[2],[3],[4],[9],[9.5],[8],[7]])
y_out = np.array([20, 22, 25, 23, -5, -8, -4, -3])
X_all = np.vstack([X, X_out])
y_all = np.concatenate([y, y_out])

models = {
    'OLS':       LinearRegression(),
    'Huber':     HuberRegressor(epsilon=1.35),
    'RANSAC':    RANSACRegressor(random_state=42),
    'Theil-Sen': TheilSenRegressor(random_state=42),
}

X_plot = np.linspace(-1, 11, 200).reshape(-1,1)
colors = ['red', 'green', 'orange', 'purple']

plt.figure(figsize=(10, 6))
plt.scatter(X.ravel(), y, s=25, color='steelblue', alpha=0.7, zorder=3, label='Clean data')
plt.scatter(X_out.ravel(), y_out, s=80, color='black', marker='x', linewidths=2, zorder=4, label='Outliers')

for (name, m), color in zip(models.items(), colors):
    m.fit(X_all, y_all)
    y_hat = m.predict(X_plot)
    plt.plot(X_plot, y_hat, color=color, linewidth=2.5, label=name)

plt.xlabel('X'); plt.ylabel('y')
plt.title('Robust regression: OLS pulled by outliers, robust methods resist')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("True slope = 2.0")
for name, m in models.items():
    if hasattr(m, 'estimator_'):   # RANSAC
        slope = m.estimator_.coef_[0]
    else:
        slope = m.coef_[0]
    print(f"  {name:<12}: slope = {slope:.4f}")


## Huber loss

```
L(r) = r²/2          if |r| ≤ ε
       ε(|r|−ε/2)    if |r| > ε
```

- For small residuals: behaves like OLS (squared loss)
- For large residuals: switches to absolute loss → limits outlier influence
- `ε = 1.35` → 95% efficiency on normal data

## RANSAC

1. Sample minimal subset of points
2. Fit a line
3. Count inliers (residual < threshold)
4. Repeat — keep the model with the most inliers

Handles extreme contamination (>30% outliers).

## Theil-Sen

β̂₁ = **median** of all (n choose 2) pairwise slopes

Breakdown point: 29.3% — still correct if up to 29.3% of data are outliers.


## When to use which

| Method | Pros | Cons |
|--------|------|------|
| OLS | Fast, exact, interpretable | Destroyed by outliers |
| Huber | Fast, efficient, tunable ε | Need to choose ε |
| RANSAC | Handles extreme contamination | Non-deterministic, slow |
| Theil-Sen | Principled, high breakdown | O(n²) — slow for large n |

**Next:** NB17 — Quantile regression.
